# Pairs Trading Strategy Backtester
**Author:** Doreen Mwangi  
**Tools:** Python · pandas · statsmodels · yfinance · matplotlib

---

## What is Pairs Trading?

Pairs trading is a **market-neutral quantitative strategy** used by hedge funds and proprietary trading desks. The core idea:

1. Find two stocks that historically move together — i.e. they are **cointegrated**
2. When the spread between them diverges significantly, **bet on mean reversion**
3. Go **long the underperformer** and **short the outperformer**
4. Exit when the spread reverts to its historical mean

Because we're simultaneously long and short, the strategy is largely **hedged against broad market moves**.

---

## Strategy Parameters

| Parameter | Value | Description |
|-----------|-------|-------------|
| Entry Z-score | ±2.0 | Open position when spread is 2 std devs from mean |
| Exit Z-score | ±0.5 | Close position when spread reverts near mean |
| Lookback window | 60 days | Rolling OLS hedge ratio estimation |
| Z-score window | 30 days | Rolling mean/std for normalisation |
| Universe | UK banks | LLOY, BARC, HSBA, NWG, STAN |

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from statsmodels.tsa.stattools import coint, adfuller
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

## 1. Load Data

We use `yfinance` to pull adjusted closing prices for a universe of UK bank stocks.
These tend to be highly correlated — good candidates for pairs trading.

In [ ]:
TICKERS = ['LLOY.L', 'BARC.L', 'HSBA.L', 'NWG.L', 'STAN.L']
START   = '2020-01-01'
END     = '2023-12-31'

raw    = yf.download(TICKERS, start=START, end=END, auto_adjust=True, progress=False)
prices = raw['Close'][TICKERS].dropna()

print(f'Downloaded {len(prices)} trading days for {len(TICKERS)} stocks.')
prices.tail()

In [ ]:
# Visualise normalised prices
norm = prices / prices.iloc[0] * 100
norm.plot(figsize=(13, 4), title='Normalised Prices (rebased to 100)')
plt.ylabel('Indexed price')
plt.grid(alpha=0.3)
plt.show()

## 2. Cointegration Analysis

**Cointegration** is different from correlation. Two stocks can be correlated short-term but not cointegrated. Cointegration means their prices share a long-run equilibrium — even if they diverge temporarily, they always revert.

We use the **Engle-Granger two-step test** from `statsmodels`:
- H₀: No cointegration (the spread is non-stationary)
- If p-value < 0.05, we reject H₀ → the pair is cointegrated

In [ ]:
def find_cointegrated_pairs(prices, threshold=0.05):
    tickers = prices.columns.tolist()
    results = []
    for t1, t2 in combinations(tickers, 2):
        _, pval, _ = coint(prices[t1], prices[t2])
        results.append({'pair': f'{t1} / {t2}', 'ticker1': t1,
                        'ticker2': t2, 'p_value': round(pval, 4),
                        'cointegrated': pval < threshold})
    return pd.DataFrame(results).sort_values('p_value')

pairs_df = find_cointegrated_pairs(prices)
print(pairs_df.to_string(index=False))

In [ ]:
# Select best (lowest p-value) cointegrated pair
cointegrated = pairs_df[pairs_df['cointegrated']]
best = cointegrated.iloc[0]
T1, T2 = best['ticker1'], best['ticker2']
print(f'Best pair: {T1} / {T2}  (p = {best["p_value"]})')

## 3. Spread & Z-score

The **spread** is computed as:
```
spread = price_A - beta * price_B
```
where `beta` is the **hedge ratio** estimated via OLS regression.

We use a **rolling window** so the hedge ratio adapts as market dynamics change.

The **Z-score** normalises the spread:
```
z = (spread - rolling_mean) / rolling_std
```

In [ ]:
LOOKBACK      = 60   # days for rolling OLS
ZSCORE_WINDOW = 30   # days for rolling mean/std

def compute_hedge_ratio(prices, t1, t2):
    y = prices[t1].values
    x = add_constant(prices[t2].values)
    return OLS(y, x).fit().params[1]

def compute_spread(prices, t1, t2, lookback=60):
    spread = pd.Series(index=prices.index, dtype=float)
    for i in range(lookback, len(prices)):
        w = prices.iloc[i-lookback:i]
        b = compute_hedge_ratio(w, t1, t2)
        spread.iloc[i] = prices[t1].iloc[i] - b * prices[t2].iloc[i]
    return spread.dropna()

def compute_zscore(spread, window=30):
    return ((spread - spread.rolling(window).mean()) / spread.rolling(window).std()).dropna()

spread = compute_spread(prices, T1, T2, LOOKBACK)
zscore = compute_zscore(spread, ZSCORE_WINDOW)

print(f'Spread computed over {len(spread)} days.')
print(f'Z-score range: {zscore.min():.2f} to {zscore.max():.2f}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

axes[0].plot(spread, color='#D4922A', linewidth=1.2)
axes[0].axhline(spread.mean(), linestyle='--', color='gray', linewidth=0.8)
axes[0].set_title(f'Spread: {T1} - beta * {T2}')
axes[0].grid(alpha=0.3)

axes[1].plot(zscore, color='#3A2A1E', linewidth=1)
for level, color, ls in [(2, '#C4623A', '--'), (-2, '#7A8C6E', '--'),
                           (0.5, '#D4922A', ':'), (-0.5, '#D4922A', ':'), (0, 'gray', '-')]:
    axes[1].axhline(level, color=color, linestyle=ls, linewidth=0.8)
axes[1].set_title('Z-score (entry at ±2, exit at ±0.5)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Backtest

Signal rules:
- Z-score > **+2.0** → **Short spread** (sell T1, buy T2)
- Z-score < **-2.0** → **Long spread** (buy T1, sell T2)  
- |Z-score| < **0.5** → **Exit position**

In [ ]:
from pairs_trading import backtest, compute_metrics

results = backtest(prices, T1, T2,
                   entry_z=2.0, exit_z=0.5,
                   lookback=LOOKBACK, zscore_window=ZSCORE_WINDOW,
                   initial_capital=100_000)

metrics = compute_metrics(results)
print('\nTrade log (first 10):')
results['trades'].head(10)

## 5. Results & Visualisation

In [ ]:
from pairs_trading import plot_results
results['prices'] = prices
plot_results(results, metrics)

## 6. Extensions & Ideas

Once this baseline is working, here are ways to extend it:

- **Stop-loss**: Exit if Z-score exceeds ±3 (spread diverging, not converging)
- **Transaction costs**: Deduct spread/commission per trade to get realistic returns
- **Portfolio of pairs**: Run multiple pairs simultaneously and weight by Sharpe
- **Kalman filter**: Replace rolling OLS with a Kalman filter for a smoother, adaptive hedge ratio
- **Sector expansion**: Try other sectors (energy, pharma, tech) and compare pair quality
- **Regime detection**: Only trade in low-volatility regimes using VIX as a filter